# 🚀 Master Orchestrator (Automated Handover Run)

Runs the **entire pipeline for real** on bundled synthetic Sentinel-1/2 scenes (seed = 42, deterministic replay):

**CRS reprojection → SAR cloud-proof mask → NDVI → Tier-2 carbon accounting → provenance embedding.**

- **Fail-Loud:** any stage failure is logged to `/quarantine/` and the run stops — bad data never silently drops.
- **Deterministic Replay:** every output GeoTIFF carries `PROVENANCE_COMMIT_SHA` + `INPUT_HASH` tags.

> Colab users: clone the repo and `%cd forest-carbon-continuity-pipeline/notebooks` first.

In [ ]:
import os
import sys

import numpy as np
import rasterio
from rasterio.transform import from_bounds

sys.path.append(os.path.abspath('../src'))

IN, OUT = '../data/input', '../data/output'
os.makedirs(IN, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print("🚀 Starting Automated Forest Carbon Continuity Pipeline...")

In [ ]:
rng = np.random.default_rng(42)
tr = from_bounds(-6.4, 52.95, -6.1, 53.15, 200, 200)


def write_wgs84(path, arr, dtype):
    with rasterio.open(path, 'w', driver='GTiff', height=200, width=200, count=1,
                       dtype=dtype, crs='EPSG:4326', transform=tr) as dst:
        dst.write(arr.astype(dtype), 1)


write_wgs84(f'{IN}/demo_wgs84.tif', rng.random((200, 200)) * 0.8 + 0.1, 'float32')
write_wgs84(f'{IN}/demo_s1_vh.tif', 0.02 + rng.random((200, 200)) * 0.06, 'float32')
write_wgs84(f'{IN}/demo_s1_vv.tif', 0.08 + rng.random((200, 200)) * 0.15, 'float32')
write_wgs84(f'{IN}/demo_s2_nir.tif', rng.random((200, 200)) * 4000 + 1000, 'uint16')
write_wgs84(f'{IN}/demo_s2_red.tif', rng.random((200, 200)) * 2500 + 200, 'uint16')

species_profile = {"driver": "GTiff", "height": 200, "width": 200, "count": 1, "dtype": "uint8",
                   "crs": "EPSG:25829",
                   "transform": from_bounds(598000, 5862000, 604000, 5868000, 200, 200)}
species = rng.choice([1, 2, 3], size=(200, 200), p=[0.5, 0.3, 0.2]).astype('uint8')
with rasterio.open(f'{OUT}/species_map.tif', 'w', **species_profile) as dst:
    dst.write(species, 1)

print("✅ Synthetic Sentinel-1/2 scenes + species map ready (seed=42, deterministic replay).")

In [ ]:
from tqdm.notebook import tqdm

from carbon_accounting import calculate_tier2_biomass
from crs_handler import reproject_to_utm29n
from metadata_embedder import compute_input_hash, embed_provenance
from quarantine import log_quarantine
from sar_optical_fusion import apply_cloud_masking, extract_ndvi_max

STEPS = [
    ('CRS reprojection to EPSG:25829',
     lambda: reproject_to_utm29n(f'{IN}/demo_wgs84.tif', f'{OUT}/01_reprojected.tif')),
    ('SAR cloud-proof forest mask',
     lambda: apply_cloud_masking(f'{IN}/demo_s1_vh.tif', f'{IN}/demo_s1_vv.tif',
                                 f'{OUT}/02_forest_mask.tif')),
    ('NDVI max composite',
     lambda: extract_ndvi_max(f'{IN}/demo_s2_nir.tif', f'{IN}/demo_s2_red.tif',
                              f'{OUT}/03_ndvi.tif')),
    ('Tier-2 carbon accounting',
     lambda: calculate_tier2_biomass(f'{OUT}/species_map.tif', f'{OUT}/04_carbon_stock.tif')),
    ('Provenance embedding',
     lambda: embed_provenance(f'{OUT}/04_carbon_stock.tif', f'{OUT}/05_carbon_final_provenance.tif',
                              compute_input_hash(f'{OUT}/04_carbon_stock.tif'))),
]

for name, step in tqdm(STEPS, desc='Pipeline'):
    try:
        result = step()
        print(f"  ✔ {name} -> {result}")
    except Exception as exc:
        log_quarantine(tile_id=name, error_code=type(exc).__name__,
                       input_hash='n/a', details=str(exc))
        raise   # Fail-loud: never continue on a broken stage

In [ ]:
final = f'{OUT}/05_carbon_final_provenance.tif'
with rasterio.open(final) as src:
    tags = src.tags()

print("🔎 Immutable audit trail embedded in", final)
for key in ('PROVENANCE_COMMIT_SHA', 'INPUT_HASH', 'PIPELINE_VERSION'):
    value = tags.get(key, 'MISSING')
    print(f"   {key} = {value[:32]}")

print()
print("✅ Pipeline Complete — outputs in data/output/:")
for fname in sorted(os.listdir(OUT)):
    print(f"   • {fname} ({os.path.getsize(os.path.join(OUT, fname)):,} bytes)")